In [1]:
import pandas as pd
import numpy as np 

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

RANDOM_STATE = 42

In [2]:
# load data
DATA_PATH = "data/raw/train.csv"

df = pd.read_csv(DATA_PATH)

In [3]:
# Binary label: risky or not
LABEL_COLS = [
    'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'
]

df["is_risky"] = (df[LABEL_COLS].sum(axis=1) > 0).astype(int) 

print(df["is_risky"].value_counts(normalize=True))

is_risky
0    0.898321
1    0.101679
Name: proportion, dtype: float64


In [4]:
# Split data into train and validation sets
X = df["comment_text"]
y = df["is_risky"]


X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

In [5]:
# Vectorize text data by TF-IDF
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_vec = vectorizer.fit_transform(X_train) 
X_val_vec = vectorizer.transform(X_val) 
# fit + transform. fit(split all sentence into tokens(or words), remove dumplicates words, get idf of each word)
# fit will change veoctorizer, but transform will not change it. so we use fit_transform for train set, and use transform for val set.
# transform must be done after fit

In [6]:
# Train baseline model
model = LogisticRegression(
    max_iter=1000, # limit number of iterations
    n_jobs=-1 # use all CPU cores
)

model.fit(X_train_vec, y_train) 
# get coefficients for each feature and bias term. 
# log odds = b + w1x1 + w2x2 + ...
 
coef_df = pd.DataFrame({
    "feature": vectorizer.get_feature_names_out(),
    "coef": model.coef_[0]
})

coef_df.sort_values("coef", ascending=False).head(10)


,feature,coef
7397,fuck,16.987862
7433,fucking,14.194190
8669,idiot,12.754708
15922,shit,12.140985
16864,stupid,11.435170
1901,ass,10.933329
2910,bullshit,10.064583
1931,asshole,9.822176
16950,suck,9.070125
2518,bitch,8.963366


In [7]:
# Evaluate model on validation set
y_pred = model.predict(X_val_vec)
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28670
           1       0.93      0.61      0.74      3245

    accuracy                           0.96     31915
   macro avg       0.94      0.80      0.86     31915
weighted avg       0.95      0.96      0.95     31915



In [8]:
val_df = pd.DataFrame({
    "comment_text": X_val, 
    "true" : y_val,
    "pred": y_pred
})


errors = val_df[val_df["true"] != val_df["pred"]]


for i, row in errors.head(10).iterrows():
    print("[comment]:", row["comment_text"])
    print("true tag:", row["true"],";","pred tag:", row["pred"])
    print("=" * 80)


[comment]: Oh so andy can have a history of this crap, yet I retaliate and I get a warning for an indef? I really love the fair, equal and level headed fashion in which you administrate this website.
true tag: 1 ; pred tag: 0
[comment]: Oppose Iset? Crazy.
true tag: 1 ; pred tag: 0
[comment]: (unless they relate to my hatred of Arabs)
true tag: 1 ; pred tag: 0
[comment]: I don't care what you say here. I don't believe one sentence anymore.
true tag: 1 ; pred tag: 0
[comment]: Boring

Why are you so boring Gail.. stop harassing me!
true tag: 1 ; pred tag: 0
[comment]: LOTHAR VON TROTHA
GOOD RIDDANCE TO HIM, HE KILLED SO MANY INNOCENT HEREROS. HE DIED FROM TYPHOID FEVER IN PAIN, CAUGHT IT FROM POISONED FOOD, DRINK!!!
MORE INFO IS AVAILABLE HERE: mywikibiz.com/Lothar_von_Trotha
true tag: 1 ; pred tag: 0
[comment]: You are really a dog .... this is the photo of my dog which you have stolen and posted on your page .. nastered....priyadarshivishal23
true tag: 1 ; pred tag: 0
[comment]: "

 p

## Error Analysis Observations
1. Short comments with implicit sarcasm.
2. Comments with rare or obsfuscated toxic words.
3. Long comments with mixed sentiment.

In [9]:
# Save vectorizer and model
import joblib
from pathlib import Path

out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(vectorizer, out_dir / "tfidf_vectorizer.joblib")
joblib.dump(model, out_dir / "baseline_lr_model.joblib")

print("Saved vectorizer and model to data/processed/")

Saved vectorizer and model to data/processed/


In [10]:
# Save validation set with predictions
val_df.to_csv("data/processed/val.csv", index=False)
print("Saved validation set:", val_df.shape)
val_df.head()

Saved validation set: (31915, 3)


,comment_text,true,pred
63958,"""\n\nThanks for helping the plot on Once Upon ...",0,0
11723,There is nothing more ambiguous in logic than ...,0,0
18282,"Go to Hell, fatso \n\nHey, Dickwad! If you ign...",1,1
145258,"""==added Justice and Development Party==\nTurk...",0,0
152834,HI\n\nI added the links and the incidents. I t...,0,0
